In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score

features = pd.read_csv('../data/features.csv')

# Drop non-feature columns
drop_cols = ['account_id','churned','district_name','region']
X = features.drop(columns=drop_cols)
y = features['churned']

X = pd.get_dummies(X, columns=['frequency'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())

(3245, 28) (812, 28) 0.10015408320493066 0.09975369458128079


In [9]:
for col in ['unemp_95', 'crimes_95']:
    X_train[col] = pd.to_numeric(X_train[col].replace('?', np.nan))
    X_test[col] = pd.to_numeric(X_test[col].replace('?', np.nan))
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# Re-verify
for col in X_train.columns:
    if X_train[col].astype(str).eq('?').any():
        print('still bad:', col)
print('done')

done


In [10]:
import mlflow

mlflow.set_experiment("berka_churn")

results = {}

# 1. Dummy baseline
with mlflow.start_run(run_name="00_dummy"):
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train, y_train)
    auc = roc_auc_score(y_test, dummy.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['dummy'] = auc

# 2. Logistic Regression
with mlflow.start_run(run_name="01_logistic_regression"):
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    logreg = LogisticRegression(max_iter=1000)
    logreg.fit(X_train_s, y_train)
    auc = roc_auc_score(y_test, logreg.predict_proba(X_test_s)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['logreg'] = auc

# 3. Random Forest
with mlflow.start_run(run_name="02_random_forest"):
    rf = RandomForestClassifier(n_estimators=200, random_state=42)
    rf.fit(X_train, y_train)
    auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['rf'] = auc

print(results)

{'dummy': 0.5, 'logreg': 0.6929624563003496, 'rf': 0.7410531826856495}


In [13]:
from xgboost import XGBClassifier

with mlflow.start_run(run_name="03_xgboost_full"):
    xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
    xgb.fit(X_train, y_train)
    auc = roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['xgb_full'] = auc

# Leakage sensitivity: does 'has_card' (card timing could leak) change the result?
leak_cols = ['has_card']
X_train_sub = X_train.drop(columns=leak_cols)
X_test_sub = X_test.drop(columns=leak_cols)

with mlflow.start_run(run_name="04_xgboost_subset_leaksensitivity"):
    xgb_sub = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')
    xgb_sub.fit(X_train_sub, y_train)
    auc = roc_auc_score(y_test, xgb_sub.predict_proba(X_test_sub)[:,1])
    mlflow.log_metric("roc_auc", auc)
    results['xgb_subset'] = auc

print(results)

{'dummy': 0.5, 'logreg': 0.6929624563003496, 'rf': 0.7410531826856495, 'xgb_full': 0.7451318167232441, 'xgb_subset': 0.7495904477208627}


In [14]:
import joblib

final_model = xgb  # xgb_full — going with the full feature set since subset didn't meaningfully help
joblib.dump(final_model, '../models/best_model.joblib')

importances = pd.Series(final_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances.head(10))

tx_amount_std                   0.070937
balance_last                    0.064886
n_orders                        0.051205
tenure_days                     0.048400
balance_mean                    0.047691
frequency_POPLATEK PO OBRATU    0.047006
balance_min                     0.043530
tx_amount_mean                  0.042585
crimes_96                       0.038696
n_entrepreneurs_per1000         0.038597
dtype: float32


In [16]:
import os
os.makedirs('../reports', exist_ok=True)

importances.to_csv('../reports/feature_importance.csv')

In [17]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_proba = final_model.predict_proba(X_test)[:,1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[719  12]
 [ 78   3]]
              precision    recall  f1-score   support

           0       0.90      0.98      0.94       731
           1       0.20      0.04      0.06        81

    accuracy                           0.89       812
   macro avg       0.55      0.51      0.50       812
weighted avg       0.83      0.89      0.85       812



In [19]:
import numpy as np

for threshold in [0.1, 0.15, 0.2, 0.3]:
    y_pred_t = (y_pred_proba >= threshold).astype(int)
    print(f"--- threshold {threshold} ---")
    print(confusion_matrix(y_test, y_pred_t))
    print(classification_report(y_test, y_pred_t, zero_division=0))

--- threshold 0.1 ---
[[557 174]
 [ 37  44]]
              precision    recall  f1-score   support

           0       0.94      0.76      0.84       731
           1       0.20      0.54      0.29        81

    accuracy                           0.74       812
   macro avg       0.57      0.65      0.57       812
weighted avg       0.86      0.74      0.79       812

--- threshold 0.15 ---
[[610 121]
 [ 53  28]]
              precision    recall  f1-score   support

           0       0.92      0.83      0.88       731
           1       0.19      0.35      0.24        81

    accuracy                           0.79       812
   macro avg       0.55      0.59      0.56       812
weighted avg       0.85      0.79      0.81       812

--- threshold 0.2 ---
[[649  82]
 [ 59  22]]
              precision    recall  f1-score   support

           0       0.92      0.89      0.90       731
           1       0.21      0.27      0.24        81

    accuracy                           0.83   

In [20]:
def risk_tier(prob):
    if prob >= 0.15:
        return 'High'
    elif prob >= 0.10:
        return 'Medium'
    else:
        return 'Low'

churn_df_test = features.loc[X_test.index, ['account_id']].copy()
churn_df_test['churn_probability'] = y_pred_proba
churn_df_test['risk_tier'] = churn_df_test['churn_probability'].apply(risk_tier)
churn_df_test['risk_tier'].value_counts()

risk_tier
Low       594
High      149
Medium     69
Name: count, dtype: int64

In [21]:
os.makedirs('../reports', exist_ok=True)
churn_df_test.to_csv('../reports/test_predictions_with_risk_tiers.csv', index=False)

In [23]:
import json
sample = X_test.iloc[0].to_dict()
print(json.dumps(sample))

{"district_id": 5, "tenure_days": 659, "tx_count": 157, "tx_amount_mean": 10734.850318471335, "tx_amount_std": 14178.903445520307, "balance_mean": 57610.20318471338, "balance_min": 300.0, "balance_last": 79337.9, "tx_count_per_year": 31.4, "has_loan": 1.0, "has_card": 1.0, "n_orders": 3.0, "order_amount_sum": 15676.3, "n_inhabitants": 95616, "n_muni_lt499": 65, "n_muni_500_1999": 30, "n_muni_2000_9999": 4, "n_muni_gt10000": 1, "n_cities": 6, "urban_ratio": 51.4, "avg_salary": 9307, "unemp_95": 3.85, "unemp_96": 4.43, "n_entrepreneurs_per1000": 118, "crimes_95": 2616.0, "crimes_96": 3040, "frequency_POPLATEK PO OBRATU": false, "frequency_POPLATEK TYDNE": false}


In [24]:
with open('../sample_request.json', 'w') as f:
    json.dump(sample, f)